In [ ]:
import numpy as np

np.random.seed(42)

In [ ]:
data = {
    'apple': 'fruit',
    'banana': 'fruit',
    'orange': 'fruit',
    'red': 'color',
    'blue': 'color',
    'green': 'color',
    'dog': 'animal',
    'cat': 'animal',
    'lion': 'animal'
}

words = list(data.keys())
categories = list(set(data.values()))

print(f"Words: {words}")
print(f"Categories: {categories}")

In [ ]:
vocabulary = sorted(list(set(''.join(words))))
vocab_size = len(vocabulary)

char_to_idx = {char: i for i, char in enumerate(vocabulary)}
idx_to_char = {i: char for i, char in enumerate(vocabulary)}

print(f"Vocabulary: {vocabulary}")
print(f"Vocabulary Size: {vocab_size}")

def word_to_vector(word, char_to_idx, vocab_size):
    vector = np.zeros(vocab_size)
    for char in word:
        if char in char_to_idx:
            vector[char_to_idx[char]] = 1
    return vector

X = np.array([word_to_vector(word, char_to_idx, vocab_size) for word in words])

category_to_idx = {cat: i for i, cat in enumerate(categories)}
y = np.zeros((len(words), len(categories)))
for i, word in enumerate(words):
    category = data[word]
    y[i, category_to_idx[category]] = 1

print("\nEncoded words (X shape):", X.shape)
print("Encoded categories (y shape):", y.shape)
print("\nFirst word ('apple') vector:", X[0])
print("First category ('fruit') vector:", y[0])

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def cross_entropy_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-12, 1 - 1e-12)
    return -np.sum(y_true * np.log(y_pred))

In [ ]:
input_size = vocab_size
hidden_size = 10
output_size = len(categories)
learning_rate = 0.1
epochs = 5000

W1 = np.random.randn(input_size, hidden_size) * 0.5
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, output_size) * 0.5
b2 = np.zeros((1, output_size))

In [ ]:
for epoch in range(epochs):
    hidden_input = np.dot(X, W1) + b1
    hidden_output = sigmoid(hidden_input)

    final_input = np.dot(hidden_output, W2) + b2
    final_output = softmax(final_input)

    loss = cross_entropy_loss(y, final_output)

    d_output = final_output - y
    dW2 = np.dot(hidden_output.T, d_output)
    db2 = np.sum(d_output, axis=0, keepdims=True)

    d_hidden = np.dot(d_output, W2.T) * sigmoid_derivative(hidden_output)
    dW1 = np.dot(X.T, d_hidden)
    db1 = np.sum(d_hidden, axis=0, keepdims=True)

    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

In [ ]:
hidden = sigmoid(np.dot(X, W1) + b1)
predictions = softmax(np.dot(hidden, W2) + b2)
predicted_labels = np.argmax(predictions, axis=1)

idx_to_category = {v: k for k, v in category_to_idx.items()}

print("\nPredictions:")
for i, word in enumerate(words):
    print(f"{word} -> Predicted: {idx_to_category[predicted_labels[i]]}, Actual: {data[word]}")